In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

print("=" * 70)
print("Random Forest Model: Separate Training & Cross-Dataset Generalization")
print("=" * 70)
print("\nThis notebook:")
print("1. Trains Random Forest on LIAR dataset")
print("2. Trains Random Forest on ISOT dataset")
print("3. Evaluates cross-dataset generalization (train on one, test on other)")
print("=" * 70)


Random Forest Model: Separate Training & Cross-Dataset Generalization

This notebook:
1. Trains Random Forest on LIAR dataset
2. Trains Random Forest on ISOT dataset
3. Evaluates cross-dataset generalization (train on one, test on other)


In [2]:
# ============================================================================
# Load Preprocessed Datasets
# ============================================================================
# NOTE: Run data/preprocess_liar.py and data/preprocess_isot.py first to create
# the preprocessed files if they don't exist yet.

print("\n" + "=" * 70)
print("Loading Preprocessed Datasets")
print("=" * 70)

# Check if preprocessed files exist
required_files = [
    'data/processed/liar_train.csv',
    'data/processed/liar_valid.csv',
    'data/processed/liar_test.csv',
    'data/processed/isot_train.csv',
    'data/processed/isot_valid.csv',
    'data/processed/isot_test.csv'
]

missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    print("\n⚠️  WARNING: Missing preprocessed files!")
    print("Please run the preprocessing scripts first:")
    print("  python data/preprocess_liar.py")
    print("  python data/preprocess_isot.py")
    print(f"\nMissing files: {missing_files}")
    raise FileNotFoundError("Preprocessed files not found. Run preprocessing scripts first.")

# Load LIAR datasets (preprocessed)
print("\nLoading LIAR datasets...")
liar_train = pd.read_csv('data/processed/liar_train.csv')
liar_valid = pd.read_csv('data/processed/liar_valid.csv')
liar_test = pd.read_csv('data/processed/liar_test.csv')

print(f"LIAR Train: {liar_train.shape}")
print(f"LIAR Valid: {liar_valid.shape}")
print(f"LIAR Test: {liar_test.shape}")
print(f"\nLIAR Train Label Distribution:\n{liar_train['label'].value_counts()}")



Loading Preprocessed Datasets

Loading LIAR datasets...
LIAR Train: (10240, 2)
LIAR Valid: (1284, 2)
LIAR Test: (1267, 2)

LIAR Train Label Distribution:
label
0    6602
1    3638
Name: count, dtype: int64


In [3]:
# Load ISOT datasets (preprocessed)
print("\nLoading ISOT datasets...")
isot_train = pd.read_csv('data/processed/isot_train.csv')
isot_valid = pd.read_csv('data/processed/isot_valid.csv')
isot_test = pd.read_csv('data/processed/isot_test.csv')

print(f"ISOT Train: {isot_train.shape}")
print(f"ISOT Valid: {isot_valid.shape}")
print(f"ISOT Test: {isot_test.shape}")
print(f"\nISOT Train Label Distribution:\n{isot_train['label'].value_counts()}")



Loading ISOT datasets...
ISOT Train: (31428, 2)
ISOT Valid: (6735, 2)
ISOT Test: (6735, 2)

ISOT Train Label Distribution:
label
0    16436
1    14992
Name: count, dtype: int64


In [4]:
# ============================================================================
# Helper Function for Model Evaluation
# ============================================================================
def evaluate_model(pipeline, x_test, y_test, dataset_name, split_name):
    """Evaluate model and return metrics."""
    print(f"\n{'='*70}")
    print(f"{dataset_name} - {split_name} Set Evaluation")
    print(f"{'='*70}")
    
    # Make predictions
    y_pred = pipeline.predict(x_test)
    y_pred_proba = pipeline.predict_proba(x_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['False', 'True']))
    
    # Print additional metrics
    print(f"\nAdditional Metrics:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}")
    
    # Print confusion matrix
    print(f"\nConfusion Matrix:")
    print(f"  [[TN={cm[0,0]:4d}, FP={cm[0,1]:4d}]")
    print(f"   [FN={cm[1,0]:4d}, TP={cm[1,1]:4d}]]")
    
    return {
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }


In [5]:
# ============================================================================
# Random Forest Pipeline Definition
# ============================================================================
# Optimizations for speed:
# - max_features=10000: Limits TF-IDF features to top 10k (reduces dimensionality)
# - n_estimators=50: Fewer trees for faster training (can increase if needed)
# - max_depth=20: Limits tree depth to prevent overfitting and speed up training
# - max_samples=0.8: Uses 80% of data per tree (faster, still good performance)

def create_rf_pipeline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            stop_words='english', 
            ngram_range=(1, 2), 
            max_features=10000
        )),
        ('clf', RandomForestClassifier(
            n_estimators=50,
            max_depth=20,
            max_samples=0.8,
            random_state=42,
            class_weight='balanced',
            n_jobs=-1
        ))
    ])


## 1. Train Random Forest on LIAR Dataset


In [6]:
# Prepare LIAR data
x_liar_train = liar_train['text'].fillna('')
y_liar_train = liar_train['label']
x_liar_valid = liar_valid['text'].fillna('')
y_liar_valid = liar_valid['label']
x_liar_test = liar_test['text'].fillna('')
y_liar_test = liar_test['label']

# Train Random Forest on LIAR
print("\n" + "=" * 70)
print("Training Random Forest on LIAR Dataset")
print("=" * 70)
print("This may take a few minutes...")

rf_liar = create_rf_pipeline()
start_time = time.time()
rf_liar.fit(x_liar_train, y_liar_train)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

# Evaluate on validation set
results_liar_valid = evaluate_model(rf_liar, x_liar_valid, y_liar_valid, "LIAR", "Validation")

# Evaluate on test set
results_liar_test = evaluate_model(rf_liar, x_liar_test, y_liar_test, "LIAR", "Test")



Training Random Forest on LIAR Dataset
This may take a few minutes...
Training completed in 0.80 seconds

LIAR - Validation Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       0.75      0.64      0.69       864
        True       0.43      0.55      0.48       420

    accuracy                           0.61      1284
   macro avg       0.59      0.60      0.58      1284
weighted avg       0.64      0.61      0.62      1284


Additional Metrics:
  Accuracy: 0.6098
  ROC-AUC:  0.6369

Confusion Matrix:
  [[TN= 550, FP= 314]
   [FN= 187, TP= 233]]

LIAR - Test Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       0.70      0.58      0.64       818
        True       0.42      0.55      0.47       449

    accuracy                           0.57      1267
   macro avg       0.56      0.56      0.56      1267
weighted avg       0.60      0.57      0.58      1267


Additi

## 2. Train Random Forest on ISOT Dataset


In [7]:
# Prepare ISOT data
x_isot_train = isot_train['text'].fillna('')
y_isot_train = isot_train['label']
x_isot_valid = isot_valid['text'].fillna('')
y_isot_valid = isot_valid['label']
x_isot_test = isot_test['text'].fillna('')
y_isot_test = isot_test['label']

# Train Random Forest on ISOT
print("\n" + "=" * 70)
print("Training Random Forest on ISOT Dataset")
print("=" * 70)
print("This may take a few minutes...")

rf_isot = create_rf_pipeline()
start_time = time.time()
rf_isot.fit(x_isot_train, y_isot_train)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

# Evaluate on validation set
results_isot_valid = evaluate_model(rf_isot, x_isot_valid, y_isot_valid, "ISOT", "Validation")

# Evaluate on test set
results_isot_test = evaluate_model(rf_isot, x_isot_test, y_isot_test, "ISOT", "Test")



Training Random Forest on ISOT Dataset
This may take a few minutes...
Training completed in 45.48 seconds

ISOT - Validation Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.99      0.99      3522
        True       0.99      1.00      0.99      3213

    accuracy                           0.99      6735
   macro avg       0.99      0.99      0.99      6735
weighted avg       0.99      0.99      0.99      6735


Additional Metrics:
  Accuracy: 0.9921
  ROC-AUC:  0.9997

Confusion Matrix:
  [[TN=3478, FP=  44]
   [FN=   9, TP=3204]]

ISOT - Test Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       1.00      0.99      0.99      3523
        True       0.99      1.00      0.99      3212

    accuracy                           0.99      6735
   macro avg       0.99      0.99      0.99      6735
weighted avg       0.99      0.99      0.99      6735


Addit

## 3. Cross-Dataset Generalization

Cross-dataset generalization tests whether the model learns general fake news patterns 
or dataset-specific features. We train on one dataset and test on the other.


### 3.1 Train on LIAR, Test on ISOT


In [8]:
# Use the model trained on LIAR to predict on ISOT test set
print("\n" + "=" * 70)
print("Cross-Dataset: Train on LIAR → Test on ISOT")
print("=" * 70)
print("This tests if a model trained on LIAR can generalize to ISOT data.")
print("If performance drops significantly, the model may be learning")
print("dataset-specific patterns rather than general fake news patterns.\n")

results_liar_to_isot = evaluate_model(rf_liar, x_isot_test, y_isot_test, 
                                       "LIAR→ISOT", "Cross-Dataset")



Cross-Dataset: Train on LIAR → Test on ISOT
This tests if a model trained on LIAR can generalize to ISOT data.
If performance drops significantly, the model may be learning
dataset-specific patterns rather than general fake news patterns.


LIAR→ISOT - Cross-Dataset Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       0.56      0.64      0.60      3523
        True       0.53      0.44      0.48      3212

    accuracy                           0.54      6735
   macro avg       0.54      0.54      0.54      6735
weighted avg       0.54      0.54      0.54      6735


Additional Metrics:
  Accuracy: 0.5440
  ROC-AUC:  0.5764

Confusion Matrix:
  [[TN=2260, FP=1263]
   [FN=1808, TP=1404]]


### 3.2 Train on ISOT, Test on LIAR


In [9]:
# Use the model trained on ISOT to predict on LIAR test set
print("\n" + "=" * 70)
print("Cross-Dataset: Train on ISOT → Test on LIAR")
print("=" * 70)
print("This tests if a model trained on ISOT can generalize to LIAR data.")
print("If performance drops significantly, the model may be learning")
print("dataset-specific patterns rather than general fake news patterns.\n")

results_isot_to_liar = evaluate_model(rf_isot, x_liar_test, y_liar_test, 
                                       "ISOT→LIAR", "Cross-Dataset")



Cross-Dataset: Train on ISOT → Test on LIAR
This tests if a model trained on ISOT can generalize to LIAR data.
If performance drops significantly, the model may be learning
dataset-specific patterns rather than general fake news patterns.


ISOT→LIAR - Cross-Dataset Set Evaluation

Classification Report:
              precision    recall  f1-score   support

       False       0.64      0.99      0.78       818
        True       0.14      0.00      0.00       449

    accuracy                           0.64      1267
   macro avg       0.39      0.50      0.39      1267
weighted avg       0.47      0.64      0.51      1267


Additional Metrics:
  Accuracy: 0.6417
  ROC-AUC:  0.4977

Confusion Matrix:
  [[TN= 812, FP=   6]
   [FN= 448, TP=   1]]


## 4. Results Summary


In [10]:
# Create summary table
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

summary_data = {
    'Model': [
        'RF on LIAR (Valid)',
        'RF on LIAR (Test)',
        'RF on ISOT (Valid)',
        'RF on ISOT (Test)',
        'LIAR→ISOT (Cross)',
        'ISOT→LIAR (Cross)'
    ],
    'Accuracy': [
        results_liar_valid['accuracy'],
        results_liar_test['accuracy'],
        results_isot_valid['accuracy'],
        results_isot_test['accuracy'],
        results_liar_to_isot['accuracy'],
        results_isot_to_liar['accuracy']
    ],
    'ROC-AUC': [
        results_liar_valid['roc_auc'],
        results_liar_test['roc_auc'],
        results_isot_valid['roc_auc'],
        results_isot_test['roc_auc'],
        results_liar_to_isot['roc_auc'],x
        results_isot_to_liar['roc_auc']
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("INTERPRETATION")
print("=" * 70)
print("\n1. Same-Dataset Performance:")
print("   - High accuracy on same dataset indicates the model can learn patterns")
print("   - Compare LIAR Test vs ISOT Test to see dataset difficulty")

print("\n2. Cross-Dataset Performance:")
print("   - If cross-dataset accuracy is much lower (< 0.7), the model is")
print("     learning dataset-specific features rather than general patterns")
print("   - If cross-dataset accuracy is reasonable (> 0.7), the model has")
print("     learned some generalizable fake news detection patterns")

print("\n3. Generalization Gap:")
liar_gap = results_liar_test['accuracy'] - results_liar_to_isot['accuracy']
isot_gap = results_isot_test['accuracy'] - results_isot_to_liar['accuracy']
print(f"   - LIAR→ISOT gap: {liar_gap:.4f}")
print(f"   - ISOT→LIAR gap: {isot_gap:.4f}")
print("   - Smaller gaps indicate better generalization")



RESULTS SUMMARY

             Model  Accuracy  ROC-AUC
RF on LIAR (Valid)  0.609813 0.636930
 RF on LIAR (Test)  0.569850 0.600300
RF on ISOT (Valid)  0.992131 0.999703
 RF on ISOT (Test)  0.991537 0.999628
 LIAR→ISOT (Cross)  0.544024 0.576373
 ISOT→LIAR (Cross)  0.641673 0.497694

INTERPRETATION

1. Same-Dataset Performance:
   - High accuracy on same dataset indicates the model can learn patterns
   - Compare LIAR Test vs ISOT Test to see dataset difficulty

2. Cross-Dataset Performance:
   - If cross-dataset accuracy is much lower (< 0.7), the model is
     learning dataset-specific features rather than general patterns
   - If cross-dataset accuracy is reasonable (> 0.7), the model has
     learned some generalizable fake news detection patterns

3. Generalization Gap:
   - LIAR→ISOT gap: 0.0258
   - ISOT→LIAR gap: 0.3499
   - Smaller gaps indicate better generalization
